# 학교 매뉴얼: 본문과 표 블록 만들기

두 PDF의 본문 Markdown과 별도로 복원한 표 Markdown을 읽어 `documents_df`(PDF 2건), `sources_df`(Markdown 4건), `blocks_df`를 만듭니다. 위에서부터 실행하세요. 원본 파일은 수정하지 않습니다.

`doc_id`는 PDF별 임시 식별자이고 `source_id`는 서로 다른 Markdown의 위치를 구분합니다. `block_id`는 소스 내부 순서 기준이라 문서가 수정되면 바뀔 수 있습니다. 확정된 팀 공통 ID나 검색용 청크는 아닙니다.

In [1]:
from pathlib import Path
import hashlib
import re

import pandas as pd
from IPython.display import display

# 노트북 파일 위치가 아니라 현재 작업 폴더부터 프로젝트 루트를 찾습니다.
cwd = Path.cwd()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "pyproject.toml").is_file() and (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"프로젝트 루트와 data/raw를 찾지 못했습니다: {cwd}")
DATA_DIR = ROOT / "data" / "raw"
print("현재 작업 폴더:", cwd, "\n입력 폴더:", DATA_DIR)

현재 작업 폴더: c:\mle-02-p2-team2\notebooks\school 
입력 폴더: c:\mle-02-p2-team2\data\raw


## 1. 문서와 파일 연결

한 PDF에 본문과 표 작업본을 각각 연결합니다. `doc_id`는 팀에서 확정한 manifest ID가 아닌 임시값입니다.

In [2]:
# 파일명을 명시해 data/raw에 다른 Markdown이 추가되어도 섞이지 않게 합니다.
document_specs = [
    {"doc_id": "school_manual", "title": "학교 감염병 예방·위기대응 매뉴얼 제3차 개정판",
     "body": "학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용).md",
     "tables": "학교_위기대응_매뉴얼_본문_표.md"},
    {"doc_id": "gyeonggi_guide", "title": "경기도 학교 감염병관리 실무가이드북",
     "body": "(본책) 경기도 학교 감염병관리 실무가이드북.md",
     "tables": "경기도_실무가이드북_본문_표.md"},
]

documents_df = pd.DataFrame([{"doc_id": d["doc_id"], "title": d["title"]} for d in document_specs])
source_rows = []
for doc in document_specs:
    for source_type in ("body", "tables"):
        path = DATA_DIR / doc[source_type]
        if not path.is_file():
            raise FileNotFoundError(f"입력 파일을 찾지 못했습니다: {path}")
        text = path.read_text(encoding="utf-8-sig")
        source_rows.append({
            "doc_id": doc["doc_id"], "source_id": f"{doc['doc_id']}:{source_type}",
            "source_type": source_type, "file_name": path.name,
            "source_path": path.relative_to(ROOT).as_posix(), "text": text,
            "char_count": len(text),
            # 변환된 Markdown 텍스트의 변경 감지용이며 원본 PDF 해시는 아닙니다.
            "content_hash": hashlib.sha256(text.encode("utf-8")).hexdigest(),
        })
sources_df = pd.DataFrame(source_rows)
assert documents_df["doc_id"].is_unique and sources_df["source_id"].is_unique
assert sources_df["text"].str.strip().ne("").all()
display(documents_df)
display(sources_df.drop(columns="text"))

,doc_id,title
0,school_manual,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판
1,gyeonggi_guide,경기도 학교 감염병관리 실무가이드북


,doc_id,source_id,source_type,file_name,source_path,char_count,content_hash
0,school_manual,school_manual:body,body,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용).md,data/raw/학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용).md,192726,a3eee7ac1598af7eeb62b480ada3264b60b1cd576a673f...
1,school_manual,school_manual:tables,tables,학교_위기대응_매뉴얼_본문_표.md,data/raw/학교_위기대응_매뉴얼_본문_표.md,62674,3b4b81822c8756cdfe7221afa7f52202c51e26e6a1d478...
2,gyeonggi_guide,gyeonggi_guide:body,body,(본책) 경기도 학교 감염병관리 실무가이드북.md,data/raw/(본책) 경기도 학교 감염병관리 실무가이드북.md,428656,f6e92f51d0555df581124b9eecf06ce5845fd7b090fb73...
3,gyeonggi_guide,gyeonggi_guide:tables,tables,경기도_실무가이드북_본문_표.md,data/raw/경기도_실무가이드북_본문_표.md,206868,e07cca4ecc078927994a7e968697198fb05c53a677d8cc...


## 2. Markdown을 순서 있는 블록으로 분리

수업의 문단·표별 블록 설계를 Markdown에 적용합니다. `text`는 각 Markdown의 원문 슬라이스이며 위치도 **해당 Markdown 기준**입니다. 빈 줄만 블록 사이에서 생략합니다. 본문의 `## PDF N페이지`는 쪽수로 읽습니다.

표는 원문에서 복원된 작업본입니다. 제목, 각주와 작업 메모를 버리지 않고 뒤에서 같은 `table_group_id`로 연결합니다. 원본 PDF의 정확한 인용 문장이라고 자동 판정하지 않습니다.

In [3]:
# 자주 쓸 정규식을 미리 준비합니다. r"..."은 역슬래시를 정규식에 그대로 전달합니다.
# PAGE는 실제 확인된 "## PDF 12페이지" 형태만 페이지 번호로 인식합니다.
PAGE = re.compile(r"^## PDF ([0-9]+)페이지\s*$")
# 제목의 # 개수와 제목 내용을 각각 그룹으로 잡아 계층 경로에 사용합니다.
HEADING = re.compile(r"^ {0,3}(#{1,6})[ \t]+(.+?)\s*$")
# 코드 펜스는 ``` 또는 ~~~가 3개 이상 반복되는 시작 줄입니다.
FENCE = re.compile(r"^ {0,3}(`{3,}|~{3,})")
# HTML 표의 시작·끝을 따로 찾아 내부의 빈 줄이나 #을 경계로 오인하지 않습니다.
HTML_TABLE = re.compile(r"^\s*<table\b", re.I)
HTML_END = re.compile(r"</table\s*>", re.I)
# 파이프 표는 머리글 다음에 | --- | --- | 같은 구분 행이 있어야 인정합니다.
SEPARATOR = re.compile(r"^\s*\|?\s*:?-{3,}:?\s*(?:\|\s*:?-{3,}:?\s*)+\|?\s*$")
# 이미지 링크, HTML 주석, Markdown 구분선은 삭제하지 않고 other로 남깁니다.
OTHER = re.compile(r"^\s*(?:!\[|<!--|(?:[-*_]\s*){3,}$)")

def markdown_blocks(text: str) -> list[dict]:
    """원문을 편집하지 않고 순서·위치가 있는 블록으로 분리한다."""
    # 줄바꿈까지 포함해서 나눠야 각 줄의 원래 문자 위치를 정확히 계산할 수 있습니다.
    lines = text.splitlines(keepends=True)
    # offsets[i]는 i번째 줄이 원문 문자열에서 시작하는 위치입니다.
    # 예: "가\n나"라면 두 번째 줄의 시작 위치는 2입니다.
    offsets = []
    pos = 0
    for line in lines:
        offsets.append(pos)
        pos += len(line)
    # blocks는 결과 목록, headings는 현재 제목 계층, page는 현재 PDF 쪽수입니다.
    blocks = []
    headings = {}
    page = None

    # 내부 함수: [start, end) 줄 범위를 원문 문자 범위로 바꿔 블록 하나를 추가합니다.
    # end는 포함하지 않는 끝 위치이므로 Python 슬라이싱과 같은 규칙입니다.
    def add(start, end, kind):
        # 마지막 블록은 다음 줄의 offset이 없으므로 원문 전체 길이를 끝으로 사용합니다.
        first = offsets[start]
        last = offsets[end] if end < len(lines) else len(text)
        # 1부터 시작하는 문서 내 순서와 B0001 같은 임시 블록 ID를 만듭니다.
        order = len(blocks) + 1
        # 원문 조각은 text[first:last] 그대로 보관해 나중에 출처로 되짚을 수 있습니다.
        blocks.append({
            "block_order": order, "block_id": f"B{order:04d}",
            "section": " > ".join(headings.values()), "kind": kind,
            "pdf_page": page, "text": text[first:last],
            "start_offset": first, "end_offset": last,
        })

    # 현재 줄에 |가 있고 바로 다음 줄이 표 구분 행인지 함께 확인합니다.
    def table_start(i):
        return i + 1 < len(lines) and "|" in lines[i] and bool(SEPARATOR.match(lines[i + 1]))

    # 일반 문단을 읽다가 다른 구조의 시작을 만나면 그 앞에서 멈추기 위한 검사입니다.
    def special(i):
        line = lines[i]
        return bool(PAGE.match(line) or HEADING.match(line) or FENCE.match(line)
                    or HTML_TABLE.match(line) or OTHER.match(line) or table_start(i))

    # i는 현재 읽는 줄 번호입니다. 각 구조를 처리하면 다음 미처리 줄로 이동합니다.
    i = 0
    while i < len(lines):
        line = lines[i]
        # 블록 사이의 빈 줄은 별도 블록으로 만들지 않습니다. 원문은 documents_df에 남습니다.
        if not line.strip():
            i += 1
            continue
        # 페이지 구분자는 먼저 처리해 일반 제목 계층에 포함되지 않게 합니다.
        match = PAGE.match(line)
        if match:
            # 정규식 첫 번째 괄호의 숫자를 정수로 바꿉니다. 이후 블록에 현재 쪽수를 붙입니다.
            page = int(match.group(1))
            add(i, i + 1, "other")
            i += 1
            continue
        # 코드 블록은 닫는 펜스까지 한 덩어리로 보존합니다.
        match = FENCE.match(line)
        if match:
            marker = match.group(1)
            # 시작과 같은 문자(` 또는 ~)가 시작 길이 이상 반복돼야 닫는 펜스입니다.
            closing = re.compile(r"^ {0,3}" + re.escape(marker[0]) + "{" + str(len(marker)) + r",}\s*$")
            end = i + 1
            while end < len(lines) and not closing.match(lines[end]):
                end += 1
            # 닫는 줄이 있으면 포함하고, 없어도 원문 끝을 넘지 않도록 합니다.
            end = min(end + 1, len(lines))
            add(i, end, "code")
            i = end
            continue
        # HTML 표는 닫는 </table>까지 줄바꿈을 유지합니다.
        if HTML_TABLE.match(line):
            end = i + 1
            while end < len(lines) and not HTML_END.search(lines[end - 1]):
                end += 1
            add(i, end, "table")
            i = end
            continue
        # Markdown 표는 머리글·구분 행을 포함하고 이어지는 | 줄을 함께 묶습니다.
        if table_start(i):
            end = i + 2
            while end < len(lines) and lines[end].strip() and "|" in lines[end]:
                end += 1
            add(i, end, "table")
            i = end
            continue
        # 제목은 자신의 레벨 이상인 기존 하위 제목을 지우고 새 제목을 기록합니다.
        match = HEADING.match(line)
        if match:
            level = len(match.group(1))
            # 예: 새 ## 제목이 나오면 이전 ##·###은 지우고 상위 #은 유지합니다.
            headings = {key: value for key, value in headings.items() if key < level}
            headings[level] = match.group(2).strip()
            add(i, i + 1, "heading")
            i += 1
            continue
        # 이미지·구분선은 한 줄, 여러 줄 HTML 주석은 닫는 표시까지 보존합니다.
        if OTHER.match(line):
            end = i + 1
            if line.lstrip().startswith("<!--") and "-->" not in line:
                while end < len(lines) and "-->" not in lines[end - 1]:
                    end += 1
            add(i, end, "other")
            i = end
            continue
        # 어느 특수 구조에도 해당하지 않으면 빈 줄 또는 다음 구조 전까지 일반 텍스트입니다.
        end = i + 1
        while end < len(lines) and lines[end].strip() and not special(end):
            end += 1
        add(i, end, "text")
        i = end
    # 순서가 있는 딕셔너리 목록을 반환합니다. 이 단계에서 LLM 청크를 만들지는 않습니다.
    return blocks

## 3. 표 제목의 쪽수와 표 묶음 연결

표 작업본의 `## PDF 21쪽 ...`, `## PDF 15페이지 ...`, `## PDF 33~34쪽 ...` 제목을 인식합니다. 제목 아래의 표와 설명은 다음 표 제목 전까지 같은 묶음입니다. 제목에 범위가 있으면 시작·끝을 모두 기록하고, 각 행이 어느 쪽에 있는지는 추정하지 않습니다.

표 주변 글에는 실제 각주와 작성자의 복원·검토 메모가 섞여 있으므로 일괄적으로 `supplement_unclassified`로 보관합니다. 표와 함께 확인하되 자동으로 근거에 포함하지 않습니다. 파일에 나타난 순서는 PDF 페이지 순서와 다를 수 있습니다.

In [4]:
# 표 제목의 PDF 쪽수와 인쇄 쪽수(선택적 범위)를 구합니다.
TABLE_PAGE = re.compile(r"^##\s+PDF\s*(\d+)(?:\s*[~～-]\s*(\d+))?\s*(?:쪽|페이지)")
PRINTED_PAGE = re.compile(r"인쇄\s*(\d+)(?:\s*[~～-]\s*(\d+))?\s*쪽")

def annotate_table_blocks(blocks, source_id):
    """원문 순서의 블록에 표 묶음, 쪽수 범위와 검토용 역할을 더한다."""
    group_no = 0
    group_id = title = None
    page_start = page_end = printed_start = printed_end = None
    for block in blocks:
        # ## 표 제목은 새 묶음의 시작입니다. 표 파일 내 번호는 임시 식별자입니다.
        heading = block["text"].strip() if block["kind"] in ("heading", "other") else ""
        match = TABLE_PAGE.match(heading)
        if match:
            group_no += 1
            group_id = f"{source_id}:G{group_no:04d}"
            title = heading.removeprefix("##").strip()
            page_start = int(match.group(1))
            page_end = int(match.group(2) or match.group(1))
            printed = PRINTED_PAGE.search(heading)
            printed_start = int(printed.group(1)) if printed else None
            printed_end = int(printed.group(2) or printed.group(1)) if printed else None
        elif block["kind"] == "heading" and heading.startswith("### ") and page_start is not None:
            # 일부 표 파일은 PDF 페이지 제목 아래 ### 표 제목을 별도로 둡니다.
            group_no += 1
            group_id = f"{source_id}:G{group_no:04d}"
            title = heading.removeprefix("###").strip()
        elif heading.startswith("# ") or heading.startswith("## "):
            # 문서 제목·검증 범위처럼 표가 아닌 제목은 이전 표에 묶지 않습니다.
            group_id = title = None
            page_start = page_end = printed_start = printed_end = None

        block.update({
            "table_group_id": group_id, "table_title": title,
            "pdf_page": page_start, "pdf_page_end": page_end,
            "printed_page": printed_start, "printed_page_end": printed_end,
            "context_role": ("table_title" if match or (heading.startswith("### ") and group_id) else
                             "table" if group_id and block["kind"] == "table" else
                             "supplement_unclassified" if group_id else "document_note"),
        })
    return blocks

block_rows = []
for source in sources_df.itertuples(index=False):
    blocks = markdown_blocks(source.text)
    if source.source_type == "tables":
        blocks = annotate_table_blocks(blocks, source.source_id)
    else:
        for block in blocks:
            # 본문 쪽수는 명시된 PDF 페이지이고 인쇄 쪽수는 추정하지 않습니다.
            block.update({"pdf_page_end": block["pdf_page"], "printed_page": None,
                          "printed_page_end": None, "table_group_id": None,
                          "table_title": None, "context_role": "body"})
    for block in blocks:
        block_rows.append({"doc_id": source.doc_id, "source_id": source.source_id,
                           "source_type": source.source_type, "file_name": source.file_name,
                           "content_hash": source.content_hash, **block})

blocks_df = pd.DataFrame(block_rows)
if blocks_df.empty:
    raise ValueError("생성된 블록이 없습니다.")
blocks_df["char_count"] = blocks_df["text"].str.len()
for column in ("pdf_page", "pdf_page_end", "printed_page", "printed_page_end"):
    blocks_df[column] = blocks_df[column].astype("Int64")
print("전체 블록:", len(blocks_df))
display(blocks_df.drop(columns="text").head())

전체 블록: 21216


,doc_id,source_id,source_type,file_name,content_hash,block_order,block_id,section,kind,pdf_page,start_offset,end_offset,pdf_page_end,printed_page,printed_page_end,table_group_id,table_title,context_role,char_count
0,school_manual,school_manual:body,body,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용).md,a3eee7ac1598af7eeb62b480ada3264b60b1cd576a673f...,1,B0001,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용),heading,<NA>,0,40,<NA>,<NA>,<NA>,NaN,NaN,body,40
1,school_manual,school_manual:body,body,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용).md,a3eee7ac1598af7eeb62b480ada3264b60b1cd576a673f...,2,B0002,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용),text,<NA>,41,383,<NA>,<NA>,<NA>,NaN,NaN,body,342
2,school_manual,school_manual:body,body,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용).md,a3eee7ac1598af7eeb62b480ada3264b60b1cd576a673f...,3,B0003,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용),text,<NA>,386,458,<NA>,<NA>,<NA>,NaN,NaN,body,72
3,school_manual,school_manual:body,body,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용).md,a3eee7ac1598af7eeb62b480ada3264b60b1cd576a673f...,4,B0004,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용),other,<NA>,459,463,<NA>,<NA>,<NA>,NaN,NaN,body,4
4,school_manual,school_manual:body,body,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용).md,a3eee7ac1598af7eeb62b480ada3264b60b1cd576a673f...,5,B0005,학교 감염병 예방·위기대응 매뉴얼 제3차 개정판(초중고·특수학교용),other,1,464,476,1,<NA>,<NA>,NaN,NaN,body,12


## 4. 원문 위치·누락 검증

같은 PDF의 본문과 표 파일은 각각 별도 텍스트이므로 `(source_id, block_id)`를 키로 검사합니다. 이 검사는 Markdown → 블록 변환만 다루며 PDF → Markdown 변환의 누락·오인식은 검증하지 않습니다.

In [5]:
# assert는 조건이 거짓이면 실행을 중단합니다.
# block_id는 파일마다 다시 B0001부터 시작하므로 source_id와 함께 중복을 검사합니다.
assert not blocks_df[["source_id", "block_id"]].duplicated().any()
# 텍스트가 누락되거나 공백뿐인 블록이 있는지 확인합니다.
# notna()는 결측값, strip()은 앞뒤 공백, ne("")는 빈 문자열이 아님을 검사합니다.
assert blocks_df["text"].notna().all() and blocks_df["text"].str.strip().ne("").all()
# 입력된 네 소스 모두 블록 생성 결과에 포함됐는지 소스 ID 집합을 비교합니다.
assert set(blocks_df["source_id"]) == set(sources_df["source_id"])
# {source_id: Markdown 원문} 형태의 사전을 만들어 블록의 출처별 원문을 찾습니다.
originals = sources_df.set_index("source_id")["text"].to_dict()
# source_id별로 묶어 각 파일의 원문과 해당 파일의 블록만 비교합니다.
for source_id, group in blocks_df.groupby("source_id", sort=False):
    original = originals[source_id]
    # 블록 순서가 1부터 빠짐없이 이어지는지 확인합니다.
    assert group["block_order"].tolist() == list(range(1, len(group) + 1))
    # 직전 블록의 끝 위치입니다. 첫 블록은 원문 시작인 0부터 비교합니다.
    previous_end = 0
    for block in group.itertuples(index=False):
        # 범위가 이전 블록과 겹치지 않고 원문 안에 있는지 확인합니다.
        # start_offset은 포함, end_offset은 포함하지 않는 Python 슬라이싱 기준입니다.
        assert previous_end <= block.start_offset < block.end_offset <= len(original), block.block_id
        # 두 블록 사이에 생략된 부분이 있다면 공백뿐이어야 합니다.
        assert not original[previous_end:block.start_offset].strip(), block.block_id
        # 기록된 위치로 원문을 다시 잘라 블록 텍스트와 정확히 같은지 확인합니다.
        assert original[block.start_offset:block.end_offset] == block.text, block.block_id
        # 다음 블록 검사에 사용할 직전 끝 위치를 갱신합니다.
        previous_end = block.end_offset
    # 마지막 블록 뒤에도 공백이 아닌 글자가 남아 있지 않아야 합니다.
    assert not original[previous_end:].strip(), source_id

# 표 작업본에서 인식한 표는 반드시 PDF 쪽수가 있는 묶음에 속해야 합니다.
# 표 파일에서 표로 인식된 블록만 골라 다음 검증과 결과 확인에 사용합니다.
tables = blocks_df[(blocks_df["source_type"] == "tables") & (blocks_df["kind"] == "table")]
# 표가 하나 이상이고, 모든 표에 묶음 ID가 있는지 확인합니다.
assert not tables.empty and tables["table_group_id"].notna().all()
# 표의 PDF 시작·끝 페이지가 기록됐는지 확인합니다. 번호의 정확성은 PDF 대조가 필요합니다.
assert tables["pdf_page"].notna().all() and tables["pdf_page_end"].notna().all()
print("소스 4개 원문·범위 검증 및 표 묶음 검증 통과")

소스 4개 원문·범위 검증 및 표 묶음 검증 통과


## 5. 결과 확인과 다음 단계

종류별 수와 표 묶음을 확인합니다. `table_group_id`로 제목·표·주변 설명을 함께 조회할 수 있습니다. 본문과 표가 같은 내용을 중복할 수 있으므로 이후 추출에서 원문 대조 후 처리 기준을 정해야 합니다.

### 문서와 소스별 블록 수

`crosstab`은 PDF 문서와 소스 유형별로 제목·본문·표 등의 블록 수를 셉니다. `All`은 합계입니다. `body`는 본문, `tables`는 별도 표 작업본입니다.

In [ ]:
display(pd.crosstab([blocks_df["doc_id"], blocks_df["source_type"]], blocks_df["kind"], margins=True))

### 블록 길이 분포

소스와 블록 종류별 문자 수의 개수(`count`), 평균(`mean`), 표준편차(`std`), 최솟값·사분위수·최댓값을 확인합니다. 유난히 긴 블록은 분리 오류인지 실제 긴 표인지 살펴볼 수 있습니다.

In [ ]:
display(blocks_df.groupby(["source_type", "kind"])["char_count"].describe().round(1))

### 표 블록의 제목과 페이지

검증 셀에서 만든 `tables`의 앞 8개를 보여 줍니다. `pdf_page`와 `pdf_page_end`는 PDF 쪽수 범위이며 `printed_page`는 표 제목에 인쇄 쪽수가 명시된 경우의 시작 쪽수입니다.

In [ ]:
display(tables[["doc_id", "table_group_id", "table_title", "pdf_page", "pdf_page_end", "printed_page"]].head(8))

### 표 묶음의 실제 내용

표 작업본마다 첫 표가 속한 묶음을 골라 제목·표·주변 설명을 원문 순서로 출력합니다. `context_role`은 블록의 역할이며, `supplement_unclassified`는 각주와 작성자 메모를 아직 구분하지 않은 설명입니다. 각 블록은 앞 500자만 보여 줍니다.

In [ ]:
for source_id, group in tables.groupby("source_id", sort=False):
    first_group = group.iloc[0]["table_group_id"]
    print(f"\n--- {source_id}: {first_group} ---")
    for block in blocks_df[blocks_df["table_group_id"] == first_group].itertuples(index=False):
        print(f"[{block.context_role}] {block.text[:500]}")

표의 내용은 수동 복원·변환된 작업본이고 검증 메모에는 전수 확인의 한계가 기록돼 있습니다. 표 주변 설명은 각주와 작업 메모를 아직 구분하지 않았으며, `table_group_id`는 파일의 제목 구간 단위라 여러 표가 함께 들어갈 수 있습니다. 표 수치·조건·인용은 PDF 원본에서 확인하세요. `pdf_page`는 PDF 파일 쪽수이고 `printed_page`는 표 제목에서 읽은 인쇄 쪽수입니다. LLM 호출·Neo4j 적재는 수행하지 않습니다.